In [1]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"


In [2]:
#____________________________ part a ______________________________#
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, accuracy_score

# Load the dataset
data = pd.read_csv('creditcard.csv')
#data.sample(10)

# Set the random seed for reproducibility
random_seed = 22
data = data.sample(n=10000, random_state=random_seed)

# Drop rows with NaN or non-numeric values
data = data.dropna()
data = data.apply(pd.to_numeric, errors='coerce')
data = data.dropna()

# Separate the features (X) and the target variable (y)
X = data.drop('Class', axis=1)
y = data['Class']

# Scale the features using StandardScaler
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Split the dataset into training and validation data
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=15)

# Define the parameter grid for grid search
param_grid = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale', 'auto']
}

# Create an SVM classifier object
svm = SVC()

# Perform grid search to find the best parameters
grid_search = GridSearchCV(svm, param_grid, scoring='accuracy', cv=5)
grid_search.fit(X_train, y_train)

# Get the best parameters and best score
best_params = grid_search.best_params_
best_score = grid_search.best_score_

print("Best Parameters:", best_params)
print("Best Score:", best_score)

# Create an SVM classifier with the best parameters
svm = SVC(**best_params)

# Fit the model on the training data
svm.fit(X_train, y_train)

# Predict the target variable for validation data
y_pred = svm.predict(X_val)

# Evaluate the performance of the model
print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_pred))

precision = precision_score(y_val, y_pred)
recall = recall_score(y_val, y_pred)
f1 = f1_score(y_val, y_pred)
accuracy = accuracy_score(y_val, y_pred)

# Print the precision, recall, F1-score, and accuracy
print("\nPrecision:", precision)
print("Recall:", recall)
print("F1-score:", f1)
print("Accuracy:", accuracy)


GridSearchCV(cv=5, estimator=SVC(),
             param_grid={'C': [0.1, 1, 10], 'gamma': ['scale', 'auto'],
                         'kernel': ['linear', 'rbf']},
             scoring='accuracy')

Best Parameters: {'C': 0.1, 'gamma': 'scale', 'kernel': 'linear'}
Best Score: 0.999625


SVC(C=0.1, kernel='linear')


Confusion Matrix:
[[1995    1]
 [   0    4]]

Precision: 0.8
Recall: 1.0
F1-score: 0.888888888888889
Accuracy: 0.9995


In [22]:
#____________________________ part b ______________________________#
from sklearn.svm import OneClassSVM

# Create an instance of the One-Class SVM
nu = 0.004  # Adjust this parameter based on the desired outlier detection rate
kernel = 'rbf'  # You can also try 'linear' or other kernels
gamma = 'scale'  # You can also try 'auto' or a specific float value

one_class_svm = OneClassSVM(nu=nu, kernel=kernel, gamma=gamma)

# Fit the model on the training data
one_class_svm.fit(X_train)

# Predict the outliers on the training data
y_pred_train = one_class_svm.predict(X_train)

# Count the number of outliers
num_outliers_train = len(y_pred_train[y_pred_train == -1])
print("Number of outliers in training data:", num_outliers_train, "\n")

# Convert the predictions to binary (1: normal, -1: outlier)
y_pred_train[y_pred_train == 1] = 0  
y_pred_train[y_pred_train == -1] = 1  

# Calculate evaluation metrics
precision = precision_score(y_train, y_pred_train)
recall = recall_score(y_train, y_pred_train)
f1 = f1_score(y_train, y_pred_train)
accuracy = accuracy_score(y_train, y_pred_train)

# Print the evaluation metrics
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)
print("Accuracy:", accuracy)


OneClassSVM(nu=0.004)

Number of outliers in training data: 187 

Precision: 0.0427807486631016
Recall: 0.6666666666666666
F1-score: 0.08040201005025126
Accuracy: 0.977125


In [23]:
#____________________________ part c ______________________________#

# Predict the target variable for the validation data
y_pred_val = svm.predict(X_val)

# Convert the predictions to binary (1: normal, -1: outlier)
y_pred_val[y_pred_val == 1] = 0  
y_pred_val[y_pred_val == -1] = 1  

# Calculate the number of outliers in the validation data
num_outliers = sum(y_pred_val)

# Calculate evaluation metrics
precision = precision_score(y_val, y_pred_val, zero_division=0)
recall = recall_score(y_val, y_pred_val)
f1 = f1_score(y_val, y_pred_val)
accuracy = accuracy_score(y_val, y_pred_val)

# Print the number of outliers and evaluation metrics
print("Number of Outliers in Validation Data:", num_outliers)
print()
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)
print("Accuracy:", accuracy)


Number of Outliers in Validation Data: 0

Precision: 0.0
Recall: 0.0
F1-score: 0.0
Accuracy: 0.998
